## masukan library yang digunakan ##

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Sisca
[nltk_data]     Cahyani\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## load datset ##

In [3]:
data = pd.read_csv('dataset_sms_spam_v1.csv')
data.head()

,teks,label
0,IM3 Ooredoo: isi ulang min 50rb hari ini dpt b...,2
1,TSEL-INFO: Paket Combo Sakti 75GB + nelpon sep...,2
2,XL Axiata khusus pelanggan setia bonus TikTok ...,2
3,Smartfren: internet unlimited nonstop mulai 40...,2
4,[PROMO] Beli paket Flash mulai 1GB di MY TELKO...,2


## Text Preprocessing ##

## Case Folding

In [4]:
import re

# membuat fungsi untuk case folding
def casefolding(text):
    text = text.lower()                                 # merubah kalimat jadi huruf kecil
    text = re.sub(r'https?://\S+|wwww\.\S+','', text)   # menghapus url dari kalimat
    text = re.sub(r'[-+]?[0-9]+', '', text)              # menghaps angka dari kalimat
    text = re.sub(r'[^\w\s]', '', text)                 # menghapus tanda baca
    text = text.strip()
    return text

In [5]:
# membandingkan before dan after case folding
raw_sample = data['teks'].iloc[2]
case_folding = casefolding(raw_sample)

print('Raw data\t : ',raw_sample)
print('Case Folding\t :', case_folding)

Raw data	 :  XL Axiata khusus pelanggan setia bonus TikTok unlimited 7 hari tanpa biaya tambahan. aktifkan via myXL sekarang
Case Folding	 : xl axiata khusus pelanggan setia bonus tiktok unlimited  hari tanpa biaya tambahan aktifkan via myxl sekarang


In [6]:
key_norm = pd.read_csv('key_norm.csv')

def text_normalize(text):
    text = ' '.join([key_norm[key_norm['singkat'] == word]['hasil'].values[0]
    if (key_norm['singkat'] == word).any()
    else word for word in text.split()                 
    ])

    text = str.lower(text)
    return text

In [7]:
# membandingkan before dan after word normalization

raw_data = data['teks'].iloc[696]
word_normal = text_normalize(raw_data)

print('Raw Data\t :', raw_data)
print('Word Normalize\t :', word_normal)

Raw Data	 : Belum de.. Saya lagi nyoba ngejar..
Word Normalize	 : belum de.. saya lagi nyoba ngejar..


## filtering (Stopword Removal)

In [8]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

stopwords_ind = stopwords.words('indonesian')

In [9]:
len(stopwords_ind)

758

In [10]:
# melihat daftar stopwords dari nltk
stopwords_ind

['ada',
 'adalah',
 'adanya',
 'adapun',
 'agak',
 'agaknya',
 'agar',
 'akan',
 'akankah',
 'akhir',
 'akhiri',
 'akhirnya',
 'aku',
 'akulah',
 'amat',
 'amatlah',
 'anda',
 'andalah',
 'antar',
 'antara',
 'antaranya',
 'apa',
 'apaan',
 'apabila',
 'apakah',
 'apalagi',
 'apatah',
 'artinya',
 'asal',
 'asalkan',
 'atas',
 'atau',
 'ataukah',
 'ataupun',
 'awal',
 'awalnya',
 'bagai',
 'bagaikan',
 'bagaimana',
 'bagaimanakah',
 'bagaimanapun',
 'bagi',
 'bagian',
 'bahkan',
 'bahwa',
 'bahwasanya',
 'baik',
 'bakal',
 'bakalan',
 'balik',
 'banyak',
 'bapak',
 'baru',
 'bawah',
 'beberapa',
 'begini',
 'beginian',
 'beginikah',
 'beginilah',
 'begitu',
 'begitukah',
 'begitulah',
 'begitupun',
 'bekerja',
 'belakang',
 'belakangan',
 'belum',
 'belumlah',
 'benar',
 'benarkah',
 'benarlah',
 'berada',
 'berakhir',
 'berakhirlah',
 'berakhirnya',
 'berapa',
 'berapakah',
 'berapalah',
 'berapapun',
 'berarti',
 'berawal',
 'berbagai',
 'berdatangan',
 'beri',
 'berikan',
 'berikut'

In [11]:
# membuat fungsi stopword removal

# menambahkan kata dalam stopwords
more_stopword = ['tsel', 'gb', 'rb', 'btw']
stopwords_ind = stopwords_ind + more_stopword

def remove_stop_word(text):
    clean_words = []
    text = text.split()
    for word in text:
        if word not in stopwords_ind:
            clean_words.append(word)
    return " ".join(clean_words)


In [12]:
raw_sample = data['teks'].iloc[696]
case_folding = casefolding(raw_sample)
stopword_removal = remove_stop_word(case_folding)

print('Raw Data \t\t :', raw_data)
print('Case Folding \t\t :', case_folding)
print('Stopword Reomoval \t\t', stopword_removal)

Raw Data 		 : Belum de.. Saya lagi nyoba ngejar..
Case Folding 		 : belum de saya lagi nyoba ngejar
Stopword Reomoval 		 de nyoba ngejar


## Stemming

In [13]:
!pip -q install sastrawi


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# merubah kata menjadi kata dasar
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# membuat fungsi untuk stemming bahasa indonesia
def stemming(text):
    text = stemmer.stem(text)
    return text

In [15]:
raw_sample = data['teks'].iloc[696]
case_folding = casefolding(raw_sample)
stopword_removal = remove_stop_word(case_folding)
text_stemming = stemming(stopword_removal)

print('Raw Data \t\t :', raw_sample)
print('Case Folding \t\t :', case_folding)
print('Stopword Removal \t\t :', stopword_removal)
print('Stemming \t\t :', text_stemming)

Raw Data 		 : Belum de.. Saya lagi nyoba ngejar..
Case Folding 		 : belum de saya lagi nyoba ngejar
Stopword Removal 		 : de nyoba ngejar
Stemming 		 : de nyoba ngejar


In [16]:
# membuat fungsi untuk menggabungkan seluruh langkah text preprocessing

def text_preprocessing_process(text) :
    text = casefolding(text)
    text = text_normalize(text)
    text = remove_stop_word(text)
    text = stemming(text)
    return text


In [17]:
%%time
data['clean_teks']= data['teks'].apply(text_preprocessing_process)

CPU times: total: 7min
Wall time: 13min 20s


In [18]:
data

,teks,label,clean_teks
0,IM3 Ooredoo: isi ulang min 50rb hari ini dpt b...,2,im ooredoo isi ulang min bonus kuota malam cek...
1,TSEL-INFO: Paket Combo Sakti 75GB + nelpon sep...,2,tselinfo paket combo sakti telpon puas rprbhr ...
2,XL Axiata khusus pelanggan setia bonus TikTok ...,2,xl axiata khusus langgan setia bonus tiktok un...
3,Smartfren: internet unlimited nonstop mulai 40...,2,smartfren internet unlimited nonstop rbbln str...
4,[PROMO] Beli paket Flash mulai 1GB di MY TELKO...,2,promo beli paket flash my telkomsel app extra ...
...,...,...,...
1191,"Aku lagi di kampus sekarang, dosennya belum da...",0,kampus dosen nunggu
1192,"Maaf baru balas, td ketiduran habis ngerjain t...",0,maaf balas tidur habis tugas kelompok pagi
1193,Tadi ibu bilang nanti malam makan di rumah nen...,0,bilang malam makan rumah nenek kumpul keluarga
1194,Besok jadi bimbingan skripsi jam 10 kan? janga...,0,besok bimbing skripsi jam lupa bawa revisi pro...


In [19]:
# simpan data yang sudah dipreprocessing ke dalam file csv
data.to_csv('clean_data.csv')

## Feature Engineering

In [20]:
# pisahkan kolom feature dan target
x = data['clean_teks']
y = data['label']

In [21]:
x

0       im ooredoo isi ulang min bonus kuota malam cek...
1       tselinfo paket combo sakti telpon puas rprbhr ...
2       xl axiata khusus langgan setia bonus tiktok un...
3       smartfren internet unlimited nonstop rbbln str...
4       promo beli paket flash my telkomsel app extra ...
                              ...                        
1191                                  kampus dosen nunggu
1192           maaf balas tidur habis tugas kelompok pagi
1193       bilang malam makan rumah nenek kumpul keluarga
1194    besok bimbing skripsi jam lupa bawa revisi pro...
1195    rumah jam an jalan macet banget gara hujan der...
Name: clean_teks, Length: 1196, dtype: object

In [22]:
y

0       2
1       2
2       2
3       2
4       2
       ..
1191    0
1192    0
1193    0
1194    0
1195    0
Name: label, Length: 1196, dtype: int64

## Feature Exraction (TF-IDF dan N-Gram)

In [23]:
# save model
import pickle

#TF IDF
from sklearn.feature_extraction.text import TfidfVectorizer

#Unigram
vec_TF_IDF  = TfidfVectorizer(ngram_range=(1,1))
vec_TF_IDF.fit(x)

x_tf_idf = vec_TF_IDF.transform(x)

pickle.dump(vec_TF_IDF.vocabulary_,open("feature_tf-idf.sav", "wb"))

In [24]:
# menampilkan vocabulary dari tif-idf
vec_TF_IDF.vocabulary_

{'im': 1203,
 'ooredoo': 2134,
 'isi': 1286,
 'ulang': 3170,
 'min': 1873,
 'bonus': 432,
 'kuota': 1610,
 'malam': 1761,
 'cek': 540,
 'aplikasi': 166,
 'myim': 1953,
 'tselinfo': 3114,
 'paket': 2168,
 'combo': 596,
 'sakti': 2611,
 'telpon': 2977,
 'puas': 2408,
 'rprbhr': 2578,
 'aktif': 66,
 'promo': 2382,
 'xl': 3442,
 'axiata': 231,
 'khusus': 1514,
 'langgan': 1639,
 'setia': 2729,
 'tiktok': 3035,
 'unlimited': 3187,
 'biaya': 380,
 'tambah': 2927,
 'via': 3227,
 'myxl': 1956,
 'smartfren': 2809,
 'internet': 1259,
 'nonstop': 2050,
 'rbbln': 2481,
 'streaming': 2868,
 'gaming': 959,
 'batas': 298,
 'favorit': 878,
 'beli': 334,
 'flash': 900,
 'my': 1951,
 'telkomsel': 2974,
 'app': 168,
 'extra': 870,
 'lte': 1716,
 'mnthr': 1901,
 'buru': 497,
 'tselmemytsel': 3118,
 'sk': 2788,
 'axis': 232,
 'warnet': 3258,
 'jam': 1319,
 'ml': 1891,
 'pubg': 2410,
 'ff': 885,
 'axisnet': 234,
 'tri': 3089,
 'indonesia': 1214,
 'guna': 1052,
 'habis': 1061,
 'ya': 3457,
 'tuhan': 3133,
 '

In [25]:
# melihat jumlah feature
print(len(vec_TF_IDF.get_feature_names_out()))

3504


In [26]:
# melihat fitur apa saja yang ada didalam corpus
print(vec_TF_IDF.get_feature_names_out())

['aa' 'aamiiiin' 'aamiin' ... 'zjt' 'zona' 'ztkm']


In [27]:
x1 =  vec_TF_IDF.transform(x).toarray()
data_tabular_tf_idf = pd.DataFrame(x1,columns=vec_TF_IDF.get_feature_names_out())
data_tabular_tf_idf

,aa,aamiiiin,aamiin,ab,abadi,abai,abbee,abdul,acara,acaratks,...,yudisium,yuk,yuks,yuni,yunit,zalora,zarkasi,zjt,zona,ztkm
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1192,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1193,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1194,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [28]:
data_tabular_tf_idf.iloc[10:20,60:70]

,akang,akangteteh,akbar,akreditasi,akses,aksi,aktif,aktifasi,aktivasi,aktivitas
10,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
11,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
12,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
14,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
15,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.510644,0.0,0.0,0.0
18,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
19,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


## Feature Selection

In [29]:
x_train = np.array(data_tabular_tf_idf)
y_train = np.array(y)

In [30]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

chi2_features = SelectKBest(chi2, k=3000)
x_kbest_features = chi2_features.fit_transform(x_train, y_train)

# untuk reduced features
print('Original Feature Number', x_train.shape[1])
print('Reduced feature Number', x_kbest_features.shape[1])

Original Feature Number 3504
Reduced feature Number 3000


In [31]:
Data = pd.DataFrame(chi2_features.scores_,columns=['Nilai'])
Data

,Nilai
0,0.875776
1,0.432501
2,1.604932
3,0.677454
4,0.749100
...,...
3499,1.102668
3500,0.497342
3501,0.677454
3502,2.850606


In [32]:
# menampilkan feature beserta nilainya

feature = vec_TF_IDF.get_feature_names_out()
feature

Data['Fitur'] = feature
Data

,Nilai,Fitur
0,0.875776,aa
1,0.432501,aamiiiin
2,1.604932,aamiin
3,0.677454,ab
4,0.749100,abadi
...,...,...
3499,1.102668,zalora
3500,0.497342,zarkasi
3501,0.677454,zjt
3502,2.850606,zona


In [33]:
# mengurutkan nilai feature terbaik
Data.sort_values(by='Nilai', ascending=False)

,Nilai,Fitur
2168,48.648500,paket
1610,45.393808,kuota
1063,44.168464,hadiah
2262,37.049033,pin
334,35.412682,beli
...,...,...
1485,0.051857,kena
1885,0.046576,minta
1578,0.041063,kopi
962,0.016392,ganti


In [34]:
data['label'].unique()

array([2, 1, 0])

In [35]:
kata = "hadiah"

for label in data['label'].unique():
    subset = data[data['label'] == label]
    muncul = subset['clean_teks'].apply(lambda t: kata in str(t).split()).sum()
    tidak_muncul = len(subset) - muncul
    print(f"Label {label} -> muncul: {muncul}, tidak muncul: {tidak_muncul}, total: {len(subset)}")

Label 2 -> muncul: 3, tidak muncul: 252, total: 255
Label 1 -> muncul: 117, tidak muncul: 237, total: 354
Label 0 -> muncul: 1, tidak muncul: 586, total: 587


In [36]:
mask = chi2_features.get_support()
mask

array([ True,  True,  True, ...,  True,  True,  True], shape=(3504,))

In [37]:
# menampilkan fitur yang terpilih berdasarkan nilai mask atau nilai tertinggi yang sudah ditetapkan pada chi square

new_feature=[]
for bool, f in zip(mask, feature):
    if bool :
        new_feature.append(f)
    selected_feature=new_feature
selected_feature

['aa',
 'aamiiiin',
 'aamiin',
 'ab',
 'abadi',
 'abai',
 'abbee',
 'abdul',
 'acara',
 'acaratks',
 'ada',
 'adapromo',
 'adi',
 'adik',
 'admin',
 'administrasi',
 'adminlte',
 'ado',
 'adrian',
 'adu',
 'aduh',
 'advertising',
 'aea',
 'aesthetic',
 'afbe',
 'affc',
 'afr',
 'afrika',
 'agam',
 'agen',
 'agendain',
 'agenpulsa',
 'ags',
 'agst',
 'agsts',
 'agt',
 'agtskinfodlj',
 'agua',
 'agun',
 'agus',
 'agust',
 'agustuskunjungi',
 'ahaha',
 'ahub',
 'aigoo',
 'air',
 'ajaa',
 'ajaaa',
 'ajabri',
 'ajak',
 'ajar',
 'ajeng',
 'akademik',
 'akang',
 'akbar',
 'akreditasi',
 'akses',
 'aksi',
 'aktif',
 'aktifasi',
 'aktivasi',
 'aktivitas',
 'akucintaislam',
 'akumulasi',
 'akun',
 'akurasi',
 'akurat',
 'alaikum',
 'alaikumsaya',
 'alaiqum',
 'alam',
 'alamat',
 'alamsyah',
 'alesannya',
 'alfagift',
 'alfamart',
 'algoritma',
 'alhamdulillah',
 'alhuda',
 'ali',
 'aliando',
 'all',
 'allah',
 'alphard',
 'alquran',
 'aman',
 'amanda',
 'ambil',
 'amin',
 'ampuun',
 'an',
 'anab

In [38]:
# membuat vocabulary baru berdasarkan fitur yang terseleksi

new_selection_feature = {}

for (k,v) in vec_TF_IDF.vocabulary_.items():
    if k in selected_feature:
        new_selection_feature[k]=v

new_selection_feature

{'im': 1203,
 'ooredoo': 2134,
 'isi': 1286,
 'ulang': 3170,
 'min': 1873,
 'bonus': 432,
 'kuota': 1610,
 'malam': 1761,
 'cek': 540,
 'aplikasi': 166,
 'myim': 1953,
 'tselinfo': 3114,
 'paket': 2168,
 'combo': 596,
 'sakti': 2611,
 'telpon': 2977,
 'puas': 2408,
 'rprbhr': 2578,
 'aktif': 66,
 'promo': 2382,
 'xl': 3442,
 'axiata': 231,
 'khusus': 1514,
 'langgan': 1639,
 'setia': 2729,
 'tiktok': 3035,
 'unlimited': 3187,
 'biaya': 380,
 'tambah': 2927,
 'myxl': 1956,
 'smartfren': 2809,
 'internet': 1259,
 'nonstop': 2050,
 'rbbln': 2481,
 'streaming': 2868,
 'gaming': 959,
 'batas': 298,
 'favorit': 878,
 'beli': 334,
 'flash': 900,
 'my': 1951,
 'telkomsel': 2974,
 'app': 168,
 'extra': 870,
 'lte': 1716,
 'mnthr': 1901,
 'buru': 497,
 'tselmemytsel': 3118,
 'sk': 2788,
 'axis': 232,
 'warnet': 3258,
 'jam': 1319,
 'ml': 1891,
 'pubg': 2410,
 'ff': 885,
 'axisnet': 234,
 'tri': 3089,
 'indonesia': 1214,
 'guna': 1052,
 'habis': 1061,
 'ya': 3457,
 'tuhan': 3133,
 'sd': 2651,
 'm

In [39]:
len(new_selection_feature)

3000

In [40]:
pickle.dump(new_selection_feature,open("new_selected_feature_tf-idf.sav","wb"))

In [41]:
# menampilkan fitur-fitur yang sudah diseleksi

data_selected_feature = pd.DataFrame(x_kbest_features, columns=selected_feature)
data_selected_feature

,aa,aamiiiin,aamiin,ab,abadi,abai,abbee,abdul,acara,acaratks,...,yudisium,yuk,yuks,yuni,yunit,zalora,zarkasi,zjt,zona,ztkm
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1192,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1193,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1194,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Modeling

In [42]:
selected_x = x_kbest_features
selected_x

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1196, 3000))

In [43]:
# import library
import random
from sklearn.model_selection import train_test_split

#import algoritma naive bayes
from sklearn.naive_bayes import MultinomialNB

In [44]:
x = selected_x
y = data.label

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [45]:
# menampilkan jumlah data training dan data testing
print('Banyaknya X_train : ', len(x_train))
print('Banyaknya X_test : ', len(x_test))
print('Banyaknya Y_train : ', len(y_train))
print('Banyaknya Y_test : ', len(y_test))


Banyaknya X_train :  956
Banyaknya X_test :  240
Banyaknya Y_train :  956
Banyaknya Y_test :  240


In [46]:
# prosess training menggunakan naive bayes
text_algorthm = MultinomialNB()

In [47]:
model = text_algorthm.fit(x_train, y_train)

In [48]:
# membuat model prediksi

data_input =("tolong belikan nama pulsa nomor as  nama teman mama celaka keluarganya hubung mama ganti uangnyapenting")
data_input = text_preprocessing_process(data_input)

# load
tfidf = TfidfVectorizer

loaded_vec = TfidfVectorizer(decode_error="replace", vocabulary=set(pickle.load(open("new_selected_feature_tf-idf.sav", "rb"))))

hasil = model.predict(loaded_vec.fit_transform([data_input]))

if(hasil==0):
    s = "SMS Normal"
elif(hasil==1):
    s = "SMS Fraud"
else:
    s = "SMS Promo"

print("Hasil Prediksi : \n", s)

Hasil Prediksi : 
 SMS Fraud


## Evaluasi Model

In [49]:
# masukan library yg dibutuhkan
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

predicted = model.predict(x_test)

CM = confusion_matrix(y_test, predicted)

print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       120
           1       0.91      0.89      0.90        76
           2       0.87      0.89      0.88        44

    accuracy                           0.93       240
   macro avg       0.91      0.91      0.91       240
weighted avg       0.93      0.93      0.93       240



In [50]:
pickle.dump(model,open("model_fraud.sav", "wb"))